# SubSight hull retrain (Kaggle)

Setup once: Settings sidebar > Accelerator = GPU T4, Internet = ON. Add the LIACI dataset as input. Then Run All.
Same baseline as tasks/hull-defect/README.md: U-Net resnet34, 256px, 20 epochs, seed 42. Expect macro ~0.25.
Weights stay in /kaggle/working. Download best.pth from the file panel right after the run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU: Settings sidebar > Accelerator > GPU'
!test -d SubSight || git clone https://github.com/manocw/SubSight.git 2>&1 | tail -1
%cd /kaggle/working/SubSight
!git pull --ff-only 2>&1 | tail -1
!pip install -q -r requirements.txt 2>&1 | tail -1

In [ ]:
# LIACI mount check. Config already points at the Kaggle mount.
# If the mount path differs, auto-detect via train_test_split.csv.
from pathlib import Path
import csv
import yaml

cfg_path = Path('tasks/hull-defect/config.yaml')
cfg = yaml.safe_load(open(cfg_path))
root = Path(cfg['data']['root'])
print('configured root:', root)

if not (root / 'train_test_split.csv').exists():
    cands = list(Path('/kaggle/input').rglob('train_test_split.csv'))
    print('candidates:', cands)
    assert cands, 'LIACI dataset not mounted. Add it under Add data.'
    root = cands[0].parent
    cfg['data']['root'] = str(root)
    yaml.safe_dump(cfg, open(cfg_path, 'w'))
    print('patched config root ->', root)
else:
    print('mount ok')

print('images:', len(list((root / 'images').glob('*'))))
print('mask dirs:', sorted(p.name for p in (root / 'masks').iterdir()))
rows = list(csv.DictReader(open(root / 'train_test_split.csv')))
tr = sum(1 for r in rows if r['split'].lower() == 'train')
te = sum(1 for r in rows if r['split'].lower() != 'train')
print(f'split rows: {tr} train / {te} test (expect 1370 / 523)')
print('classes:', cfg['data']['classes'])

In [ ]:
!python tasks/hull-defect/src/train.py

In [ ]:
!python tasks/hull-defect/src/evaluate.py --checkpoint tasks/hull-defect/checkpoints/best.pth --num-images 4 --out tasks/hull-defect/outputs/eval.png
!ls -lh tasks/hull-defect/checkpoints/best.pth tasks/hull-defect/outputs/eval.png